In [1]:
# ── Core libraries ─────────────────────────────────────────────────
import numpy as np
import pandas as pd
import os
import re
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# ── Scikit-learn ───────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
    roc_auc_score,
)
from sklearn.utils.class_weight import compute_sample_weight

# ── XGBoost (Step 1 — new import) ─────────────────────────────────
import xgboost as xgb
from xgboost import XGBClassifier

# ── SMOTE for class imbalance (Step 1 — new import) ───────────────
from imblearn.over_sampling import SMOTE

# ── SHAP for model explainability (Step 1 — new import) ──────────
import shap

# ── Visualization ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── New Dependencies for Tuning & Ensembling (Step 1 Additions) ─────
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING) # Keep output clean

from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.model_selection import train_test_split
import lightgbm as lgb
from lightgbm import LGBMClassifier

print("All imports loaded successfully.")
print(f"  xgboost : {xgb.__version__}")
print(f"  shap    : {shap.__version__}")

All imports loaded successfully.
  xgboost : 1.7.6
  shap    : 0.49.1


In [2]:
# ── Load dataset ──────────────────────────────────────────────────
LOCAL_PATH = 'ai4i2020.csv'
KAGGLE_PATH = '/kaggle/input/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020/ai4i2020.csv'

if os.path.exists(LOCAL_PATH):
    df = pd.read_csv(LOCAL_PATH)
    print(f"Loaded from local path: {LOCAL_PATH}")
elif os.path.exists(KAGGLE_PATH):
    df = pd.read_csv(KAGGLE_PATH)
    print(f"Loaded from Kaggle path: {KAGGLE_PATH}")
else:
    raise FileNotFoundError(
        f"Dataset not found at '{LOCAL_PATH}' or '{KAGGLE_PATH}'. "
        "Please place ai4i2020.csv in the notebook directory."
    )

print(f"Shape: {df.shape}")
df.head()

Loaded from local path: ai4i2020.csv
Shape: (10000, 14)


,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


In [3]:
# ── Feature engineering (same as original) ────────────────────────
df['temperature_difference'] = df['Process temperature [K]'] - df['Air temperature [K]']
df['Power'] = np.round(
    df['Torque [Nm]'] * df['Rotational speed [rpm]'] * 2 * np.pi / 60, 4
)

df.sample(5)

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF,temperature_difference,Power
2655,2656,M17515,M,299.8,309.4,1751,27.0,173,0,0,0,0,0,0,9.6,4950.8359
8544,8545,H37958,H,298.5,309.4,2011,21.0,59,0,0,0,0,0,0,10.9,4422.4200
5558,5559,L52738,L,302.5,311.9,1553,40.1,187,0,0,0,0,0,0,9.4,6521.4542
2142,2143,L49322,L,299.3,308.9,1376,60.3,163,0,0,0,0,0,0,9.6,8688.8913
6587,6588,M21447,M,301.6,310.7,1813,24.6,210,0,0,0,0,0,0,9.1,4670.4801


In [4]:
# ── Build multi-class target ──────────────────────────────────────
failure_cols = ['TWF', 'HDF', 'PWF', 'OSF', 'RNF']
failure_names = ['No Failure', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF']

def assign_failure_class(row):
    """Return the first active failure mode (1-5), or 0 for no failure."""
    for idx, col in enumerate(failure_cols, start=1):
        if row[col] == 1:
            return idx
    return 0

df['failure_class'] = df.apply(assign_failure_class, axis=1)

print("Class distribution:")
print(df['failure_class'].value_counts().sort_index())
print()
for i, name in enumerate(failure_names):
    count = (df['failure_class'] == i).sum()
    print(f"  {i} — {name}: {count} ({count / len(df) * 100:.1f}%)")

Class distribution:
failure_class
0    9652
1      46
2     115
3      91
4      78
5      18
Name: count, dtype: int64

  0 — No Failure: 9652 (96.5%)
  1 — TWF: 46 (0.5%)
  2 — HDF: 115 (1.1%)
  3 — PWF: 91 (0.9%)
  4 — OSF: 78 (0.8%)
  5 — RNF: 18 (0.2%)


In [5]:
# ── Prepare features (X) and target (y) ───────────────────────────
drop_cols = ['UDI', 'Product ID', 'Machine failure'] + failure_cols + ['failure_class']
X = df.drop(columns=drop_cols).copy()
y = df['failure_class'].copy()

# Encode 'Type' column: L=0, M=1, H=2
X['Type'] = X['Type'].map({'L': 0, 'M': 1, 'H': 2})

# ── Clean column names ────────────────────────────────────────────
# XGBoost rejects feature names containing [ ] or <
X.columns = [re.sub(r'[\[\]<>]', '', col).strip() for col in X.columns]

print(f"Features shape: {X.shape}")
print(f"Target shape:   {y.shape}")
print(f"Clean feature names: {X.columns.tolist()}")
X.head()

Features shape: (10000, 8)
Target shape:   (10000,)
Clean feature names: ['Type', 'Air temperature K', 'Process temperature K', 'Rotational speed rpm', 'Torque Nm', 'Tool wear min', 'temperature_difference', 'Power']


,Type,Air temperature K,Process temperature K,Rotational speed rpm,Torque Nm,Tool wear min,temperature_difference,Power
0,1,298.1,308.6,1551,42.8,0,10.5,6951.5906
1,0,298.2,308.7,1408,46.3,3,10.5,6826.7227
2,0,298.1,308.5,1498,49.4,5,10.4,7749.3875
3,0,298.2,308.6,1433,39.5,7,10.4,5927.5047
4,0,298.2,308.7,1408,40.0,9,10.5,5897.8166


In [6]:
# ── Split ─────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, random_state=42, test_size=0.2, stratify=y
)

print(f"Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")
print(f"\nTrain class distribution:\n{y_train.value_counts().sort_index()}")
print(f"\nTest class distribution:\n{y_test.value_counts().sort_index()}")

Train size: 8000  |  Test size: 2000

Train class distribution:
failure_class
0    7722
1      37
2      92
3      73
4      62
5      14
Name: count, dtype: int64

Test class distribution:
failure_class
0    1930
1       9
2      23
3      18
4      16
5       4
Name: count, dtype: int64


In [7]:
# ── Scale features ────────────────────────────────────────────────
feature_names = X.columns.tolist()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Keep as DataFrames for SHAP later
X_train_scaled = pd.DataFrame(X_train_scaled, columns=feature_names, index=X_train.index)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=feature_names, index=X_test.index)

print("Scaling complete.")
X_train_scaled.head()

Scaling complete.


,Type,Air temperature K,Process temperature K,Rotational speed rpm,Torque Nm,Tool wear min,temperature_difference,Power
5154,-0.742474,2.104202,2.297306,0.169396,-0.245192,-1.300915,-0.802624,-0.103460
4921,-0.742474,1.703230,1.553067,-1.396390,1.477696,-0.845388,-1.102581,1.032727
8214,-0.742474,-0.401874,0.335221,-0.816882,0.576185,-1.096713,1.297078,0.363061
1563,-0.742474,-0.903090,-1.153258,-0.755588,1.097058,0.521194,0.097249,1.128408
180,-0.742474,-0.903090,-1.153258,-0.348818,0.986873,-0.955342,0.097249,1.334665


In [8]:
# ── Apply SMOTE (Step 3) ──────────────────────────────────────────
USE_SAMPLE_WEIGHTS = False  # Will flip to True if SMOTE fails
sample_weights = None

try:
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)
    print("✅ SMOTE applied successfully.")
    print(f"   Before SMOTE: {X_train_scaled.shape[0]} samples")
    print(f"   After  SMOTE: {X_train_resampled.shape[0]} samples")
    print(f"\n   Resampled class distribution:")
    print(f"   {pd.Series(y_train_resampled).value_counts().sort_index().to_dict()}")
except Exception as e:
    print(f"⚠️ SMOTE failed: {e}")
    print("   Falling back to sample_weight='balanced' for XGBoost.")
    USE_SAMPLE_WEIGHTS = True
    X_train_resampled = X_train_scaled
    y_train_resampled = y_train
    sample_weights = compute_sample_weight('balanced', y_train)

✅ SMOTE applied successfully.
   Before SMOTE: 8000 samples
   After  SMOTE: 46332 samples

   Resampled class distribution:
   {0: 7722, 1: 7722, 2: 7722, 3: 7722, 4: 7722, 5: 7722}


In [ ]:
# ── Step 2A — Hyperparameter Tuning (Run Once) ──────────────────────
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from catboost import CatBoostClassifier # New Import

# 1. Split the SMOTE-resampled training data
X_tune, X_val, y_tune, y_val = train_test_split(
    X_train_resampled, y_train_resampled, test_size=0.2, random_state=42, stratify=y_train_resampled
)
num_classes = len(y.unique())

# Prepare sample weights for tuning if SMOTE failed
sample_weights_tune = compute_sample_weight('balanced', y_tune) if USE_SAMPLE_WEIGHTS else None
fit_params = {'sample_weight': sample_weights_tune} if USE_SAMPLE_WEIGHTS else {}

# --- Define Objective Functions ---
def objective_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'max_depth': trial.suggest_int('max_depth', 4, 12), # Depth constrained
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'random_state': 42,
        'n_jobs': -1
    }
    model = RandomForestClassifier(**params)
    model.fit(X_tune, y_tune, **fit_params)
    return f1_score(y_val, model.predict(X_val), average='macro')

def objective_xgb(trial):
    params = {
        'objective': 'multi:softprob',
        'eval_metric': 'mlogloss',
        'num_class': num_classes,
        'tree_method': 'hist', 
        'device': 'cuda',      
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 7), # Depth constrained
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'random_state': 42,
        'use_label_encoder': False,
    }
    model = XGBClassifier(**params)
    model.fit(X_tune, y_tune, **fit_params)
    return f1_score(y_val, model.predict(X_val), average='macro')

def objective_lgb(trial):
    params = {
        'objective': 'multiclass',
        'num_class': num_classes,
        'device': 'gpu',       
        'n_estimators': trial.suggest_int('n_estimators', 100, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 7), # Depth constrained
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'random_state': 42,
        'verbose': -1,
    }
    model = LGBMClassifier(**params)
    model.fit(X_tune, y_tune, **fit_params)
    return f1_score(y_val, model.predict(X_val), average='macro')

def objective_cat(trial):
    params = {
        'loss_function': 'MultiClass',
        'task_type': 'GPU', # Enables GPU acceleration
        'iterations': trial.suggest_int('iterations', 100, 300),
        'depth': trial.suggest_int('depth', 4, 8), # CatBoost sweet spot
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'random_seed': 42,
        'verbose': 0
    }
    model = CatBoostClassifier(**params)
    # CatBoost expects sample weights directly in fit if needed
    model.fit(X_tune, y_tune, **fit_params)
    
    # CatBoost sklearn wrapper returns (N, 1) matrix sometimes, flatten ensures f1_score works
    preds = model.predict(X_val).flatten() 
    return f1_score(y_val, preds, average='macro')

# --- Run Optuna Studies ---
N_TRIALS = 30  

print("Tuning Random Forest (CPU)...")
study_rf = optuna.create_study(direction="maximize")
study_rf.optimize(objective_rf, n_trials=N_TRIALS)

print("\nTuning XGBoost (GPU)...")
study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS)

print("\nTuning LightGBM (GPU)...")
study_lgb = optuna.create_study(direction="maximize")
study_lgb.optimize(objective_lgb, n_trials=N_TRIALS)

print("\nTuning CatBoost (GPU)...")
study_cat = optuna.create_study(direction="maximize")
study_cat.optimize(objective_cat, n_trials=N_TRIALS)

# --- Print Copy-Pasteable Results ---
print("\n" + "="*50)
print("COPY AND PASTE THESE DICTIONARIES INTO STEP 2B")
print("="*50)
print(f"rf_params = {study_rf.best_params}")
print(f"xgb_params = {study_xgb.best_params}")
print(f"lgb_params = {study_lgb.best_params}")
print(f"cat_params = {study_cat.best_params}")
print("="*50)

Tuning Random Forest (CPU)...

Tuning XGBoost (GPU)...
[10:48:43] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "device" } are not used.

[10:48:45] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "device" } are not used.

[10:48:46] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "device" } are not used.

[10:48:48] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Parameters: { "device" } are not used.

[10:48:49] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0fdc6d574b9c0d168-1\xgboost\xgboost-ci-windows\src\learner.cc:767: 
Param

In [15]:
# ── Step 2B — Hardcoded Ensemble Training ───────────────────────────
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# 🛠️ THE BULLETPROOF FIX: Patch the CLASSES directly so clone() doesn't wipe them
XGBClassifier._estimator_type = "classifier"
LGBMClassifier._estimator_type = "classifier"
CatBoostClassifier._estimator_type = "classifier"

num_classes = len(y.unique())

# 1. Paste your Optuna results here
rf_params = {'n_estimators': 106, 'max_depth': 12, 'min_samples_split': 2}
xgb_params = {'n_estimators': 257, 'max_depth': 7, 'learning_rate': 0.27105986989031644}
lgb_params = {'n_estimators': 231, 'max_depth': 6, 'learning_rate': 0.16620391318835553}
cat_params = {'iterations': 256, 'depth': 4, 'learning_rate': 0.29999514114684045}

# 2. Instantiate the models with the hardcoded params + static requirements
best_rf = RandomForestClassifier(
    **rf_params, 
    random_state=42, 
    n_jobs=-1
)

best_xgb = XGBClassifier(
    **xgb_params, 
    objective='multi:softprob', 
    eval_metric='mlogloss', 
    num_class=num_classes, 
    tree_method='hist', 
    device='cuda',
    random_state=42, 
    use_label_encoder=False
)

best_lgb = LGBMClassifier(
    **lgb_params, 
    objective='multiclass', 
    num_class=num_classes, 
    device='gpu', 
    random_state=42, 
    verbose=-1
)

best_cat = CatBoostClassifier(
    **cat_params,
    loss_function='MultiClass',
    task_type='GPU',
    random_seed=42,
    verbose=0
)

# 3. Create the 4-Model Voting Ensemble
ensemble_model = VotingClassifier(
    estimators=[
        ('rf', best_rf), 
        ('xgb', best_xgb), 
        ('lgb', best_lgb), 
        ('cat', best_cat)
    ],
    voting='soft'
)

# 4. Train the Final Ensemble on the FULL SMOTE dataset
print("Training final 4-Model Ensemble on the full SMOTE-resampled dataset...")
fit_params_full = {'sample_weight': sample_weights} if USE_SAMPLE_WEIGHTS else {}

try:
    ensemble_model.fit(X_train_resampled, y_train_resampled, **fit_params_full)
except TypeError:
    ensemble_model.fit(X_train_resampled, y_train_resampled)

# Seamless handoff to Step 4
xgb_model = ensemble_model 
print("✅ Final Ensemble model training complete!")

Training final 4-Model Ensemble on the full SMOTE-resampled dataset...


ValueError: The estimator XGBClassifier should be a classifier.

In [ ]:
# ── Predictions ───────────────────────────────────────────────────
y_pred = xgb_model.predict(X_test_scaled)
y_pred_proba = xgb_model.predict_proba(X_test_scaled)

# ── Classification Report ─────────────────────────────────────────
print("=" * 70)
print("CLASSIFICATION REPORT")
print("=" * 70)
print(classification_report(
    y_test, y_pred,
    target_names=failure_names,
    zero_division=0
))

# ── Confusion Matrix ──────────────────────────────────────────────
print("=" * 70)
print("CONFUSION MATRIX")
print("=" * 70)
cm = confusion_matrix(y_test, y_pred)
print(cm)

# ── Key Metrics (Step 4 — upgraded) ───────────────────────────────
acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')

try:
    roc_auc = roc_auc_score(y_test, y_pred_proba, multi_class='ovr')
except ValueError as e:
    # Can happen if a class has 0 test samples
    roc_auc = float('nan')
    print(f"\n⚠️ ROC-AUC could not be computed: {e}")

print("\n" + "=" * 70)
print("SUMMARY METRICS")
print("=" * 70)
print(f"  Accuracy        : {acc:.4f}")
print(f"  Macro F1-Score  : {macro_f1:.4f}")
print(f"  ROC-AUC (OVR)   : {roc_auc:.4f}")

In [ ]:
# ── Confusion matrix heatmap ──────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=failure_names,
    yticklabels=failure_names,
    ax=ax
)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — XGBoost Multi-Class')
plt.tight_layout()
plt.show()

In [ ]:
# ── SHAP explainability for Ensemble (Step 5) ──────────────────────
# Note: SHAP's TreeExplainer doesn't natively support VotingClassifier, 
# so we extract the underlying models, calculate their SHAP values, and average them.

import shap
import numpy as np

# Extract underlying fitted models from our VotingClassifier
rf_model = xgb_model.named_estimators_['rf']
xgb_tree = xgb_model.named_estimators_['xgb']
lgb_model = xgb_model.named_estimators_['lgb']

print("Computing SHAP values for Random Forest...")
shap_rf = shap.TreeExplainer(rf_model).shap_values(X_test_scaled, check_additivity=False)

print("Computing SHAP values for XGBoost...")
shap_xgb = shap.TreeExplainer(xgb_tree).shap_values(X_test_scaled)

print("Computing SHAP values for LightGBM...")
shap_lgb = shap.TreeExplainer(lgb_model).shap_values(X_test_scaled)

def standardize_shap(sv, num_classes):
    """Standardize SHAP output format across different model libraries."""
    if isinstance(sv, list):
        return np.array(sv)
    # If returned as a single 3D array: (n_samples, n_features, num_classes)
    elif isinstance(sv, np.ndarray) and len(sv.shape) == 3:
        # Swap axes to match the list array structure: (num_classes, n_samples, n_features)
        return np.transpose(sv, (2, 0, 1))
    return sv

std_rf = standardize_shap(shap_rf, num_classes)
std_xgb = standardize_shap(shap_xgb, num_classes)
std_lgb = standardize_shap(shap_lgb, num_classes)

# Because we are using Soft Voting (probability averaging),
# averaging the underlying SHAP arrays perfectly mirrors the ensemble's logic.
avg_shap = (std_rf + std_xgb + std_lgb) / 3.0

# Re-wrap into list format for the downstream SHAP summary plot code to read
shap_values = [avg_shap[i] for i in range(num_classes)]

print(f"\n✅ Averaged SHAP values computed for {X_test_scaled.shape[0]} test samples.")
print(f"   Number of output classes: {len(shap_values)}")
print(f"   Shape per class: {shap_values[0].shape}")

In [ ]:
# ── SHAP summary plot (all classes) ───────────────────────────────
print("SHAP Summary Plot — Feature Importance Across All Failure Modes")
print("=" * 70)

shap.summary_plot(
    shap_values,
    X_test_scaled,
    class_names=failure_names,
    plot_type='bar',
    show=True
)

In [ ]:
# ── SHAP detailed dot plot per class ──────────────────────────────
if isinstance(shap_values, list):
    for cls_idx, cls_name in enumerate(failure_names):
        if cls_idx < len(shap_values):
            print(f"\nSHAP values for class: {cls_name}")
            shap.summary_plot(
                shap_values[cls_idx],
                X_test_scaled,
                plot_type='dot',
                show=True
            )
else:
    # For binary or when shap returns a single array
    shap.summary_plot(
        shap_values,
        X_test_scaled,
        plot_type='dot',
        show=True
    )